# Credit Card Fraud Detection Pipeline
### Final Presentation Version
This notebook contains the complete pipeline for credit card fraud detection using Decision Trees and Support Vector Classifiers.

In [ ]:
# 1. Import Necessary Libraries
import math
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from pylab import rcParams

# Model Selection & Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine Learning Classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.naive_bayes import GaussianNB

# Evaluation Metrics
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Inline plotting for Jupyter Notebooks
%matplotlib inline

# Global Configuration
rcParams['figure.figsize'] = 14, 8
RANDOM_SEED = 42
LABELS = ["Normal", "Fraud"]

In [ ]:
# 2. Data Loading & Exploratory Data Analysis (EDA)
# Load dataset (Ensure this path is correct based on your environment)
dataset = pd.read_csv("/content/creditcard.csv.zip")

print("--- Dataset Head ---")
print(dataset.head())

print("\n--- Dataset Info ---")
print(dataset.info())

# Check for missing values across the dataset
has_nulls = dataset.isnull().values.any()
print(f"\nAre there missing values? {has_nulls}")

# Separate normal and fraud data for summary statistics
fraud_data = dataset[dataset['Class'] == 1]
normal_data = dataset[dataset['Class'] == 0]
print(f"Fraud instances shape: {fraud_data.shape} | Normal instances shape: {normal_data.shape}")

print("\n--- Fraud Transactions Amount Statistics ---")
print(fraud_data.Amount.describe())

print("\n--- Normal Transactions Amount Statistics ---")
print(normal_data.Amount.describe())

In [ ]:
# 3. Data Visualization
# Plot 1: Target Class Distribution
plt.figure()
set_class = pd.value_counts(dataset['Class'], sort=True)
set_class.plot(kind='bar', rot=0)
plt.title("Class Distribution of Transaction")
plt.xticks(range(2), LABELS)
plt.xlabel("Classes")
plt.ylabel("No of occurrences")
plt.show()

# Plot 2: Feature Correlation Heatmap
plt.figure(figsize=(20, 20))
corrmat = dataset.corr()
top_corr_features = corrmat.index
sns.heatmap(dataset[top_corr_features].corr(), annot=True, cmap="RdYlGn", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# 4. Feature Extraction & Train-Test Split
x = dataset.iloc[:, 1:30].values
y = dataset.iloc[:, 30].values

print(f"\nInput Feature Shape: {x.shape}")
print(f"Output Target Shape: {y.shape}")
print(f"Class Labels Sample: {y[:10]}...")

# Split data into train and test sets
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=0)

print(f"\nxtrain shape: {xtrain.shape}")
print(f"xtest shape  : {xtest.shape}")
print(f"ytrain shape : {ytrain.shape}")
print(f"ytest shape  : {ytest.shape}")

# Feature Standardization
stdsc = StandardScaler()
xtrain = stdsc.fit_transform(xtrain)
xtest = stdsc.transform(xtest)

print(f"\nFirst standardized training example sample:\n {xtrain[0][:5]}... (truncated)")

In [ ]:
# 5. Model 1: Decision Tree Classifier
print("\n" + "="*50)
print("TRAINING MODEL 1: DECISION TREE")
print("="*50)

dt_classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
dt_classifier.fit(xtrain, ytrain)
y_pred_decision_tree = dt_classifier.predict(xtest)

# Metrics calculation
com_decision = confusion_matrix(ytest, y_pred_decision_tree)
print("Confusion Matrix:\n", com_decision)

accuracy_dt = ((com_decision[0][0] + com_decision[1][1]) / com_decision.sum()) * 100
error_rate_dt = ((com_decision[0][1] + com_decision[1][0]) / com_decision.sum()) * 100
precision_dt = (com_decision[1][1] / (com_decision[1][1] + com_decision[0][1])) * 100
recall_dt = (com_decision[1][1] / (com_decision[1][1] + com_decision[1][0])) * 100

print(f"Accuracy Decision Tree   : {accuracy_dt:.2f}%")
print(f"Error Rate Decision Tree : {error_rate_dt:.2f}%")
print(f"Precision (True Fake)    : {precision_dt:.2f}%")
print(f"Recall (Sensitivity)     : {recall_dt:.2f}%")

print("\nDetailed Classification Report (Decision Tree):")
print(classification_report(ytest, y_pred_decision_tree, target_names=LABELS))

In [ ]:
# 6. Model 2: Support Vector Classifier (SVC)
print("\n" + "="*50)
print("TRAINING MODEL 2: SUPPORT VECTOR MACHINE")
print("="*50)

svc_classifier = SVC(kernel='rbf', random_state=0)
svc_classifier.fit(xtrain, ytrain)
y_pred_svc = svc_classifier.predict(xtest)

# Metrics calculation
cm2 = confusion_matrix(ytest, y_pred_svc)
print("Confusion Matrix:\n", cm2)

accuracy_svc = ((cm2[0][0] + cm2[1][1]) / cm2.sum()) * 100
error_rate_svc = ((cm2[0][1] + cm2[1][0]) / cm2.sum()) * 100
precision_svc = (cm2[1][1] / (cm2[1][1] + cm2[0][1])) * 100 if (cm2[1][1] + cm2[0][1]) > 0 else 0.0
recall_svc = (cm2[1][1] / (cm2[1][1] + cm2[1][0])) * 100 if (cm2[1][1] + cm2[1][0]) > 0 else 0.0

print(f"Accuracy SVC   : {accuracy_svc:.2f}%")
print(f"Error Rate SVC : {error_rate_svc:.2f}%")
print(f"Precision SVC  : {precision_svc:.2f}%")
print(f"Recall SVC     : {recall_svc:.2f}%")

print("\nDetailed Classification Report (SVC):")
print(classification_report(ytest, y_pred_svc, target_names=LABELS))